In [ ]:
import joblib
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report

pd.set_option("display.max_columns", None)

In [ ]:
merged_df = pd.read_parquet("merged_df_with_ae.parquet")
recon_error_col_names = joblib.load("recon_error_col_names.joblib")
anomaly_threshold = joblib.load("anomaly_threshold.joblib")
train_mask = joblib.load("train_mask.joblib")
ae_train_mask = joblib.load("ae_train_mask.joblib")
SENSOR_COLUMNS = joblib.load("sensor_columns.joblib")
engine_state_col_names = joblib.load("engine_state_col_names.joblib")
SENSOR_FAULT_COLUMNS = joblib.load("sensor_fault_columns.joblib")

In [ ]:
# XGBoost trains on rows the AE has SCORED but never TRAINED on — i.e. train-split
# rows that are NOT part of the AE's healthy-only training subset. This avoids
# XGBoost just re-learning what the AE already memorized as "normal."

xgb_calibration_mask = train_mask & (~ae_train_mask)

print(f"Calibration rows: {xgb_calibration_mask.sum()} (out of {train_mask.sum()} total train rows)")

In [ ]:
# Features: raw sensor state + engine context + AE's per-channel and aggregate error signal
XGB_FEATURE_COLUMNS = (
    SENSOR_COLUMNS + engine_state_col_names + recon_error_col_names + ["ae_recon_error", "ae_anomaly_flag"]
)

# Target: per-channel sensor-fault class (NONE/BIAS/DRIFT/NOISE/STUCK/DROPOUT).
# One classifier per channel — keeps each model simple, keeps SHAP interpretable
# per-channel, and avoids forcing a single joint label space across channels
# that don't actually share a fault mechanism.
SENSOR_FAULT_CLASSES = ["NONE", "BIAS", "DRIFT", "NOISE", "STUCK", "DROPOUT"]

X_xgb_train = merged_df.loc[xgb_calibration_mask, XGB_FEATURE_COLUMNS].values
X_xgb_val = merged_df.loc[~train_mask, XGB_FEATURE_COLUMNS].values  # held-out val runs

print(f"{len(XGB_FEATURE_COLUMNS)} input features")
print(f"{len(SENSOR_FAULT_COLUMNS)} target channels, {len(SENSOR_FAULT_CLASSES)} classes each")

In [ ]:
# Worth checking before trusting anything downstream: if NONE is rare in the
# calibration set for a given channel, the model won't have learned "when not
# to flag" nearly as well as "when to flag."

for col in SENSOR_FAULT_COLUMNS:
    counts = merged_df.loc[xgb_calibration_mask, col].value_counts().sort_index()
    print(f"{col}: {dict(zip(SENSOR_FAULT_CLASSES, counts.reindex(range(6), fill_value=0)))}")

In [ ]:
xgb_models = {}

for channel_col in SENSOR_FAULT_COLUMNS:
    y_train_channel = merged_df.loc[xgb_calibration_mask, channel_col].values
    y_val_channel = merged_df.loc[~train_mask, channel_col].values

    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(SENSOR_FAULT_CLASSES),
        eval_metric="mlogloss",
        early_stopping_rounds=20,
    )

    model.fit(
        X_xgb_train,
        y_train_channel,
        eval_set=[(X_xgb_val, y_val_channel)],
        verbose=False,
    )

    xgb_models[channel_col] = model
    print(f"Trained model for {channel_col} (best iteration: {model.best_iteration})")

In [ ]:
for channel_col, model in xgb_models.items():
    y_val_channel = merged_df.loc[~train_mask, channel_col].values
    y_pred = model.predict(X_xgb_val)

    print(f"\n=== {channel_col} ===")
    print(
        classification_report(
            y_val_channel,
            y_pred,
            labels=list(range(len(SENSOR_FAULT_CLASSES))),
            target_names=SENSOR_FAULT_CLASSES,
            zero_division=0,
        )
    )

In [ ]:
import shap

# Example: inspect the first sensor-fault classifier specifically —
# swap for whichever channel matters most to your demo/report.
# NOTE: main_batch_1000 only injects sensor faults on one channel
# (sensor_fault_active_cht_c3), so SENSOR_FAULT_COLUMNS has a single entry
# here; this picks it generically rather than hardcoding the channel name.
inspect_channel = SENSOR_FAULT_COLUMNS[0]
inspect_model = xgb_models[inspect_channel]

explainer = shap.TreeExplainer(inspect_model)
shap_values = explainer.shap_values(X_xgb_train[:500])  # sample for speed

shap.summary_plot(
    shap_values,
    X_xgb_train[:500],
    feature_names=XGB_FEATURE_COLUMNS,
    class_names=SENSOR_FAULT_CLASSES,
)

In [ ]:
joblib.dump(xgb_models, "xgb_sensor_fault_models.joblib")
joblib.dump(XGB_FEATURE_COLUMNS, "xgb_feature_columns.joblib")
joblib.dump(SENSOR_FAULT_CLASSES, "sensor_fault_classes.joblib")

print(f"Saved {len(xgb_models)} per-channel XGBoost models")

In [ ]:
# Since ae_anomaly_flag is one of XGBoost's own input features, this checks
# whether XGBoost is doing real discrimination beyond just echoing that flag —
# the ablation a reviewer would ask for.

for channel_col, model in xgb_models.items():
    val_mask = ~train_mask
    ae_flag_val = merged_df.loc[val_mask, "ae_anomaly_flag"].values
    y_true = merged_df.loc[val_mask, channel_col].values
    y_pred = model.predict(X_xgb_val)

    naive_agreement = (ae_flag_val.astype(bool) == (y_true != 0)).mean()
    xgb_agreement = (y_pred == y_true).mean()

    print(f"{channel_col}: naive AE-flag accuracy={naive_agreement:.3f} | XGBoost accuracy={xgb_agreement:.3f}")